## Setup

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer
from sklearn.decomposition import PCA

## Load

In [83]:
gateways_file = ('data/raw/Speed_Test_Interview_Project_Gateways_Table.csv')
speed_file = ('data/raw/Speed_Test_Interview_Project_Metrics_Table_v2.csv')

gateways_df = pd.read_csv(gateways_file)
speed_df = pd.read_csv(speed_file)
speed_df = speed_df.drop(columns=['local_hour'])

# Encode target and time
speed_df['Slow'] = speed_df['router_speedtest_tcp_64_download_Mbps_avg'] < 50
OUTPUT_COLUMN = 'Slow'
TARGET_COLUMN = 'router_speedtest_tcp_64_download_Mbps_avg' # column name or None
TIME_COLUMN = 'slot_timestamp'   # column name for time axis, or None

print(f'Shape: {gateways_df.shape}')
print(f'Shape: {speed_df.shape}')

Shape: (70319, 8)
Shape: (68119, 19)


C:\Users\Jacob\AppData\Local\Temp\ipykernel_49472\793190400.py:5: DtypeWarning: Columns (0: link_failure_detected) have mixed types. Specify dtype option on import or set low_memory=False.
  speed_df = pd.read_csv(speed_file)


In [84]:
# Fix datatypes to handle numeric IDs
numToStr1 = ['gateway_id','satellite_id','gateway_site_id']
for colName in numToStr1:
    gateways_df[colName] = gateways_df[colName].astype(str)

numToStr2 = ['cell_id','home_pop_id','gateway_id','satellite_id', 'ku_beam_target_cell_id']
for colName in numToStr2:
    speed_df[colName] = speed_df[colName].astype(str)

# Ensure timestamps are datetime and sorted
speed_df[TIME_COLUMN] = pd.to_datetime(speed_df[TIME_COLUMN])
gateways_df[TIME_COLUMN] = pd.to_datetime(gateways_df[TIME_COLUMN])
speed_df = speed_df.sort_values(TIME_COLUMN).reset_index(drop=True)
gateways_df = gateways_df.sort_values(TIME_COLUMN).reset_index(drop=True)

In [85]:
# Split holdout
choose_randomly = True
split = 0.1
if TIME_COLUMN is not None and not choose_randomly:
    speed_df = speed_df.sort_values(TIME_COLUMN)
    i = int(len(speed_df) * (1 - split))
    speed_holdout_df = speed_df.iloc[i:]
    speed_df = speed_df.iloc[:i]
else:
    speed_df, speed_holdout_df = train_test_split(speed_df, test_size = split, random_state = 1)
speed_holdout_df = speed_holdout_df.drop(columns = TARGET_COLUMN)
speed_holdout_df.to_csv('data/raw/holdout.csv', index=False)

In [86]:
# Nulls
def filter_or_flag_nulls(df, df_name, threshold_pct=1.0):
    print(f'\n{df_name} Missing Values:')
    high_null_cols = []

    for col in df.columns:
        n_null = df[col].isnull().sum()
        if n_null > 0:
            pct = 100 * n_null / len(df)
            if pct >= threshold_pct:
                print(f'{col:30s} {n_null:6d} ({pct:5.1f}%)')
                high_null_cols.append(col)
            else:
                print(f'{col:30s} {n_null:6d} ({pct:5.1f}%)')

    dropna_cols = [col for col in df.columns if col not in high_null_cols]
    df_clean = df.dropna(subset=dropna_cols)

    print(f'  Rows before: {len(df):,} → after: {len(df_clean):,}')
    return df_clean


speed_df    = filter_or_flag_nulls(speed_df,    'Speed_df')
gateways_df = filter_or_flag_nulls(gateways_df, 'Gateways_df')


Speed_df Missing Values:
ut_ping_loss_rate                 152 (  0.2%)
link_failure_detected           39722 ( 64.8%)
  Rows before: 61,307 → after: 61,155

Gateways_df Missing Values:
ka_beam_avg_util_percent           21 (  0.0%)
ka_beam_max_util_percent           21 (  0.0%)
ka_beam_active_count               27 (  0.0%)
  Rows before: 70,319 → after: 70,282


## Encode features

In [87]:
gw_features = ['ka_beam_max_util_percent', 'ka_beam_avg_util_percent', 
               'ka_beam_active_count', 'online_user_count']

speed_df = speed_df.sort_values(TIME_COLUMN).reset_index(drop=True)
gateways_df = gateways_df.sort_values(TIME_COLUMN).reset_index(drop=True)

# Get nearest gateway entry per gateway_id
gw_nearest = gateways_df[['gateway_id', 'slot_timestamp'] + gw_features].copy()
gw_nearest = gw_nearest.rename(columns={'slot_timestamp': 'gw_timestamp'})

# Backward merge (up to 60s in the past)
merge_back = pd.merge_asof(
    speed_df[[TIME_COLUMN, 'gateway_id']],
    gw_nearest,
    left_on=TIME_COLUMN,
    right_on='gw_timestamp',
    by='gateway_id',
    direction='backward',
    tolerance=pd.Timedelta(seconds=60),
)
merge_back['_offset'] = (merge_back[TIME_COLUMN] - merge_back['gw_timestamp']).dt.total_seconds().abs()

# Forward merge (up to 15s in the future)
merge_fwd = pd.merge_asof(
    speed_df[[TIME_COLUMN, 'gateway_id']],
    gw_nearest,
    left_on=TIME_COLUMN,
    right_on='gw_timestamp',
    by='gateway_id',
    direction='forward',
    tolerance=pd.Timedelta(seconds=15),
)
merge_fwd['_offset'] = (merge_fwd[TIME_COLUMN] - merge_fwd['gw_timestamp']).dt.total_seconds().abs()

# Pick the closer match per row
use_fwd = merge_fwd['_offset'].fillna(np.inf) < merge_back['_offset'].fillna(np.inf)
merged = merge_back.copy()
merged.loc[use_fwd, gw_features + ['gw_timestamp', '_offset']] = merge_fwd.loc[use_fwd, gw_features + ['gw_timestamp', '_offset']].values

# Join results back to speed_df
speed_df['gateway_entry_offset_seconds'] = (
    speed_df[TIME_COLUMN] - merged['gw_timestamp']
).dt.total_seconds()
speed_df['has_nearest_gateway_entry'] = merged['gw_timestamp'].notna().astype(int)

for feat in gw_features:
    speed_df[f'{feat}_nearest'] = merged[feat].fillna(merged[feat].median())

speed_df['gateway_entry_offset_seconds'] = speed_df['gateway_entry_offset_seconds'].fillna(-1)

new_cols = ([f'{f}_nearest' for f in gw_features] + 
            ['gateway_entry_offset_seconds', 'has_nearest_gateway_entry'])
print(f'Added {len(new_cols)} gateway recency features: {new_cols}')
print(f'Match rate: {speed_df["has_nearest_gateway_entry"].mean():.3f}')
print(f'speed_df shape: {speed_df.shape}')

Added 6 gateway recency features: ['ka_beam_max_util_percent_nearest', 'ka_beam_avg_util_percent_nearest', 'ka_beam_active_count_nearest', 'online_user_count_nearest', 'gateway_entry_offset_seconds', 'has_nearest_gateway_entry']
Match rate: 0.927
speed_df shape: (61155, 25)


In [88]:
speed_df['snr_margin'] = speed_df['ut_snr'] - speed_df['expected_min_snr']

# Clean the link failure column
speed_df['link_failure_detected'] = speed_df['link_failure_detected'].fillna('Unknown').astype(str)

# Encode gateway_site_id into speed_df
gateway_site_keys = gateways_df[['gateway_id', 'gateway_site_id']].drop_duplicates()
speed_df = speed_df.merge(gateway_site_keys, on='gateway_id', how='left')

# Encode pct of beam resource used by the user
speed_df['beam_pct_allocation'] = 1 / (speed_df['number_of_ku_beam_targets'] * speed_df['number_of_users_on_ku_beam_target'])

speed_df['users_per_gateway_beam_recent'] = speed_df['online_user_count_recent'] / speed_df['ka_beam_active_count_recent']


KeyError: 'online_user_count_recent'

In [ ]:
# Running aggregate tables
keys = ['cell_id', 'ku_beam_target_cell_id', 'home_pop_id', 'satellite_id', 
        'gateway_id', 'gateway_site_id', 'country_code']

speed_df[TIME_COLUMN] = pd.to_datetime(speed_df[TIME_COLUMN])
speed_df = speed_df.sort_values(TIME_COLUMN)

running_tables = {}

for key in keys:
    # Per-(key, time) period aggregates
    period_agg = (
        speed_df.groupby([key, TIME_COLUMN])
        .agg(**{f'median_speed_{key}': (TARGET_COLUMN, 'median'),
                f'pct_slow_{key}':     (OUTPUT_COLUMN, lambda x: 100 * x.mean())})
        .reset_index()
        .sort_values([key, TIME_COLUMN])
    )
    # Expanding historical mean, shifted so time t only sees periods *before* t
    for col in [f'median_speed_{key}', f'pct_slow_{key}']:
        period_agg[col] = (
            period_agg.groupby(key)[col]
            .transform(lambda x: x.expanding().mean().shift(1))
        )
    running_tables[key] = period_agg

# Global fallbacks
global_fallbacks = {
    'global_median_speed': speed_df[TARGET_COLUMN].median(),
    'global_pct_slow': speed_df[OUTPUT_COLUMN].mean() * 100,
}

# Merge running stats
for key in keys:
    speed_df = speed_df.merge(running_tables[key], on=[key, TIME_COLUMN], how='left')
    speed_df[f'median_speed_{key}'] = speed_df[f'median_speed_{key}'].fillna(global_fallbacks['global_median_speed'])
    speed_df[f'pct_slow_{key}'] = speed_df[f'pct_slow_{key}'].fillna(global_fallbacks['global_pct_slow'])

print(f'speed_df shape after running tables: {speed_df.shape}')
print(f'Global fallbacks: {global_fallbacks}')

speed_df shape after running tables: (61155, 41)
Global fallbacks: {'global_median_speed': np.float64(161.5406189), 'global_pct_slow': np.float64(5.911209222467501)}


In [ ]:
# Running aggregate tables
gateways_df['slot_timestamp'] = pd.to_datetime(gateways_df['slot_timestamp'])
gateways_df = gateways_df.sort_values('slot_timestamp')

gateway_keys = ['gateway_id', 'satellite_id', 'gateway_site_id']

gateway_agg_features = {
    'median_users':                    ('online_user_count',        'median'),
    'median_ka_beam_avg_util_percent': ('ka_beam_avg_util_percent', 'median'),
    'p95_ka_beam_max_util_percent':    ('ka_beam_max_util_percent', lambda x: 100 * (x > 95).mean()),
    'p100_ka_beam_max_util_percent':   ('ka_beam_max_util_percent', lambda x: 100 * (x == 100).mean()),
    'avg_ka_beam_active_count':        ('ka_beam_active_count',     'mean'),
}

running_tables_2 = {}

for key in gateway_keys:
    period_agg = (
        gateways_df.groupby([key, 'slot_timestamp'])
        .agg(**gateway_agg_features)
        .reset_index()
        .sort_values([key, 'slot_timestamp'])
    )
    rename_map = {feat: f'{feat}_{key}' for feat in gateway_agg_features.keys()}
    period_agg = period_agg.rename(columns=rename_map)

    feat_cols = [f'{feat}_{key}' for feat in gateway_agg_features.keys()]
    for col in feat_cols:
        period_agg[col] = (
            period_agg.groupby(key)[col]
            .transform(lambda x: x.expanding().mean().shift(1))
        )
    running_tables_2[key] = period_agg

# Merge gateway running stats
for key in gateway_keys:
    rt = running_tables_2[key].sort_values('slot_timestamp')
    speed_df = speed_df.sort_values(TIME_COLUMN)
    speed_df = pd.merge_asof(
        speed_df,
        rt,
        left_on=TIME_COLUMN,
        right_on='slot_timestamp',
        by=key,
        direction='backward'
    )
    # Clean up duplicate timestamp column from merge
    if 'slot_timestamp_y' in speed_df.columns:
        speed_df = speed_df.drop(columns=['slot_timestamp_y'])
    if 'slot_timestamp_x' in speed_df.columns:
        speed_df = speed_df.rename(columns={'slot_timestamp_x': TIME_COLUMN})

# Fill NaN gateway features with column medians
for key in gateway_keys:
    for feat in gateway_agg_features.keys():
        col = f'{feat}_{key}'
        if col in speed_df.columns:
            speed_df[col] = speed_df[col].fillna(speed_df[col].median())

print(f'speed_df shape after gateway running tables: {speed_df.shape}')

speed_df shape after gateway running tables: (61155, 56)


## Construct model data

In [ ]:
# Construct aggregate df — features are already merged into speed_df
agg_df = speed_df.copy()

agg_df = agg_df.drop(columns=[
    'gateway_id', 'satellite_id', 'gateway_site_id', 'cell_id', 'home_pop_id', 
    'ku_beam_target_cell_id', 'country_code', 'test_id'
])
agg_df.to_csv('data/processed/agg_df.csv', index=False)

print(agg_df.columns)

# Export running tables and fallbacks for modeling
import pickle
pickle.dump(gateway_site_keys, open('gateway_site_keys.pkl', 'wb'))
pickle.dump(running_tables, open('running_tables.pkl', 'wb'))
pickle.dump(running_tables_2, open('running_tables_2.pkl', 'wb'))
pickle.dump(global_fallbacks, open('global_fallbacks.pkl', 'wb'))

Index(['router_speedtest_tcp_64_download_Mbps_avg', 'slot_timestamp',
       'ut_ping_loss_rate', 'ut_snr', 'expected_min_snr', 'ut_latency',
       'link_failure_detected', 'ut_sat_elevation_angle_mean',
       'number_of_ku_beam_targets', 'number_of_users_on_ku_beam',
       'number_of_users_on_ku_beam_target', 'Slow',
       'gateway_entry_offset_seconds', 'has_nearest_gateway_entry',
       'ka_beam_max_util_percent_nearest', 'ka_beam_avg_util_percent_nearest',
       'ka_beam_active_count_nearest', 'online_user_count_nearest',
       'beam_pct_allocation', 'median_speed_cell_id', 'pct_slow_cell_id',
       'median_speed_ku_beam_target_cell_id',
       'pct_slow_ku_beam_target_cell_id', 'median_speed_home_pop_id',
       'pct_slow_home_pop_id', 'median_speed_satellite_id',
       'pct_slow_satellite_id', 'median_speed_gateway_id',
       'pct_slow_gateway_id', 'median_speed_gateway_site_id',
       'pct_slow_gateway_site_id', 'median_speed_country_code',
       'pct_slow_country_co

In [ ]:
# Tree-based model df
tree_df = agg_df.copy()

# drop id cols from tree_df (not agg_df)
tree_df = tree_df.drop(columns=['router_speedtest_tcp_64_download_Mbps_avg', 'slot_timestamp'])

# Export
tree_df.to_csv('data/processed/tree/tree_df.csv', index=False)

tree_df.dtypes

ut_ping_loss_rate                                  float64
ut_snr                                             float64
expected_min_snr                                   float64
ut_latency                                         float64
link_failure_detected                                  str
ut_sat_elevation_angle_mean                        float64
number_of_ku_beam_targets                            int64
number_of_users_on_ku_beam                           int64
number_of_users_on_ku_beam_target                    int64
Slow                                                  bool
gateway_entry_offset_seconds                       float64
has_nearest_gateway_entry                            int64
ka_beam_max_util_percent_nearest                   float64
ka_beam_avg_util_percent_nearest                   float64
ka_beam_active_count_nearest                       float64
online_user_count_nearest                          float64
beam_pct_allocation                                float

In [ ]:
# PCA
pca_df = agg_df.copy()
pca_df = pd.get_dummies(pca_df, columns=['link_failure_detected'], drop_first=False, dtype='int64')

y = pca_df[OUTPUT_COLUMN].astype(int)
X = pca_df.drop(columns=[TARGET_COLUMN, TIME_COLUMN, OUTPUT_COLUMN])

# Normalize
scaler = PowerTransformer(method='yeo-johnson')
X_scaled = scaler.fit_transform(X)

# Explore: fit with all components first
pca_explore = PCA()
pca_explore.fit(X_scaled)

# Find n_components for 95% variance
cumulative_var = np.cumsum(pca_explore.explained_variance_ratio_)
n_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f'Components for 95% variance: {n_95} / {X.shape[1]}')
print(f'Explained variance (first 10): {pca_explore.explained_variance_ratio_[:10].round(4)}')

# Apply PCA with optimal n_components
pca_reduced = PCA(n_components=n_95)
X_reduced = pca_reduced.fit_transform(X_scaled)

# Create output df
pca_out = pd.DataFrame(X_reduced, columns=[f'PC{i+1}' for i in range(n_95)])
pca_out[OUTPUT_COLUMN] = y
pca_out.to_csv('data/processed/pca_df.csv', index=False)

print(f'Output shape: {pca_out.shape}')

import pickle
pickle.dump(scaler, open('pca_scaler.pkl', 'wb'))
pickle.dump(pca_reduced, open('pca_model.pkl', 'wb'))

Components for 95% variance: 27 / 47
Explained variance (first 10): [0.2707 0.0781 0.0628 0.0564 0.0491 0.0423 0.0379 0.032  0.0297 0.029 ]
Output shape: (61155, 28)
